# Практика · Скрапінг сторінок

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · ДЗ: [homework.html](homework.html)

**Мережа не потрібна:** усі сторінки віддає наш власний сервер, який зошит піднімає
сам на твоєму компʼютері. Жодного походу в інтернет тут немає — і жодного чужого
сайту ми не турбуємо.

Наскрізний приклад той самий, що в темах 22 і 28, — меню маленької кавʼярні.
Тільки тепер його ніхто не віддає у зручному вигляді: є лише сторінки для людей.

Що зробимо:

1. піднімемо власний сервер, який віддає HTML-сторінки меню з пагінацією;
2. подивимось, чому регулярний вираз ламається там, де парсер не ламається;
3. розберемо сторінку через `BeautifulSoup`: `find`, `find_all`, `select`;
4. дістанемо текст і атрибути — і побачимо різницю `tag["href"]` проти `tag.get("href")`;
5. пройдемо всі сторінки за посиланням «далі» й зберемо список словників;
6. перевіримо **придатність** даних: у меню навмисно є позиція без ціни, позиція
   з зайвими пробілами й позиція з ціною в іншому форматі;
7. запишемо результат у CSV;
8. зламаємо верстку й подивимось, який селектор виживе, а який мовчки поверне нуль;
9. подивимось, чому на динамічній сторінці список порожній — і де шукати прихований JSON;
10. прочитаємо `robots.txt` і зробимо збирання ввічливим.

## 1 · Що саме ми скрапимо

Скрапити чужий сайт у навчальному зошиті не можна з двох причин, і обидві серйозні.
Етична: сотня запитів із чужого скрипта — це навантаження на сервіс, який нас ні про
що не просив. Інженерна: чужа сторінка зміниться, і зошит, який сьогодні дає вісім
позицій, за півроку дасть нуль або впаде.

Тому ми піднімемо **власний** сервер. Він віддає справжній HTTP по справжньому
TCP-зʼєднанню — `requests` не помітить різниці. Змінюється лише адреса: замість
чужого домену `127.0.0.1`, тобто цей самий компʼютер.

In [ ]:
import csv
import http.server
import json
import re
import shutil
import tempfile
import threading
import time
from pathlib import Path

import requests
from bs4 import BeautifulSoup

print("requests       :", requests.__version__)
print("beautifulsoup4 :", __import__("bs4").__version__)

Ось дані меню — те, що на справжньому сайті лежало б у базі. Три позиції з восьми
навмисно «криві», і саме на них тримається половина цього зошита:

* **Флет-вайт** — ціна із зайвими пробілами всередині тексту;
* **Раф** — ціни немає взагалі, тега `.price` у розмітці не буде;
* **Чай травʼяний** — ціна в іншому форматі (`30,00 UAH` замість `30 грн`),
  а посилання «докладніше» без адреси.

In [ ]:
MENU = [
    {"id": 1, "name": "Еспресо",       "price": "25 грн",     "cat": "кава",  "link": True},
    {"id": 2, "name": "Капучино",      "price": "45 грн",     "cat": "кава",  "link": True},
    {"id": 3, "name": "Латте",         "price": "50 грн",     "cat": "кава",  "link": True},
    {"id": 4, "name": "Флет-вайт",     "price": "  55  грн ", "cat": "кава",  "link": True},
    {"id": 5, "name": "Раф",           "price": None,         "cat": "кава",  "link": True},
    {"id": 6, "name": "Чай травʼяний", "price": "30,00 UAH",  "cat": "чай",   "link": False},
    {"id": 7, "name": "Матча",         "price": "65 грн",     "cat": "чай",   "link": True},
    {"id": 8, "name": "Какао",         "price": "40 грн",     "cat": "какао", "link": True},
]

PER_PAGE = 3                      # по три позиції на сторінку — щоб була пагінація
PAGE_COUNT = 3                    # 8 позицій = 3 + 3 + 2

print("позицій у меню:", len(MENU), "· сторінок:", PAGE_COUNT)
print("без ціни:", [item["name"] for item in MENU if item["price"] is None])

Тепер функції, які збирають HTML сторінки. Параметр `layout` знадобиться далі:
ним ми будемо **навмисно ламати верстку**, як це робить будь-який редизайн.

* `"normal"` — звичайна розмітка;
* `"wrap"` — між `<article>` і його вмістом зʼявилась зайва обгортка `<div class="row">`;
* `"rename"` — клас `name` перейменували на `title`.

In [ ]:
def item_html(item, layout):
    """Один пункт меню як шматок HTML."""
    name_class = "title" if layout == "rename" else "name"
    inner = ['<h3 class="{}">{}</h3>'.format(name_class, item["name"])]
    # ціни може не бути взагалі — тоді тега .price у розмітці просто немає
    if item["price"] is not None:
        inner.append('<span class="price">{}</span>'.format(item["price"]))
    inner.append('<span class="cat">{}</span>'.format(item["cat"]))
    if item["link"]:
        inner.append('<a class="more" href="/item?id={}">докладніше</a>'.format(item["id"]))
    else:
        # посилання-заглушка: тег є, атрибута href немає
        inner.append('<a class="more">докладніше</a>')

    if layout == "wrap":
        body = ('      <div class="row">\n'
                + "\n".join("        " + line for line in inner)
                + "\n      </div>")
    else:
        body = "\n".join("      " + line for line in inner)
    return '    <article class="item">\n' + body + "\n    </article>"


def page_html(number, layout="normal"):
    """Сторінка меню з посиланням «далі», якщо вона не остання."""
    chunk = MENU[(number - 1) * PER_PAGE : number * PER_PAGE]
    items = "\n".join(item_html(item, layout) for item in chunk)
    pager = ""
    if number < PAGE_COUNT:
        pager = ('  <nav class="pager">'
                 '<a class="next" href="/page?n={}">далі</a></nav>\n'.format(number + 1))
    return (
        '<!doctype html>\n<html lang="uk">\n<head><meta charset="utf-8">\n'
        '<title>Меню кав\'ярні · сторінка {}</title></head>\n<body>\n'
        '  <h1 class="head">Меню кав\'ярні</h1>\n'
        '  <div class="menu">\n{}\n  </div>\n{}'
        '  <p class="note">Сторінка {} з {}</p>\n</body>\n</html>\n'
    ).format(number, items, pager, number, PAGE_COUNT)


print(page_html(2))

Подивись на цей текст уважно — саме його побачить `requests`. Друга сторінка й обрана
навмисно: у ній сидять усі три «нерівності». Зверни увагу, що в позиції «Раф»
тега `<span class="price">` **немає взагалі** — це не порожній тег, це відсутній тег.
Різниця буде дуже відчутною.

### Риштування: сервер

Наступна клітинка — **риштування**, а не матеріал теми. Навчальний сервер описано
через **клас**, а класи будуть аж у [темі 30](../30-classes/lecture.html). Зараз його
не треба розуміти — просто запусти. Важливо тут лише одне: далі `requests` робитиме
до нього справжні запити.

Два рядки в класі обовʼязкові: `protocol_version = "HTTP/1.1"` вмикає постійні
зʼєднання, а `disable_nagle_algorithm = True` знімає затримку в 40 мс на кожній
відповіді. Метод `log_message` заглушений, інакше зошит заріс би рядками журналу.

In [ ]:
DYNAMIC_PAGE = """<!doctype html>
<html lang="uk">
<head><meta charset="utf-8"><title>Меню кавʼярні · нова версія</title></head>
<body>
  <h1 class="head">Меню кавʼярні</h1>
  <div class="menu" id="menu-root"></div>
  <script src="/static/menu.js"></script>
</body>
</html>
"""

MENU_JS = """fetch("/api/menu.json")
  .then(function (response) { return response.json(); })
  .then(function (items) { drawMenu(items); });
"""

ROBOTS_TXT = """User-agent: *
Disallow: /admin
Disallow: /item
Allow: /page
Crawl-delay: 1
"""

server_hits = {"count": 0}        # скільки разів сервер узагалі щось віддав


class MenuHandler(http.server.BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"      # без цього постійні зʼєднання не працюють
    disable_nagle_algorithm = True     # без цього кожна відповідь чекає ~40 мс

    def log_message(self, *args):      # глушимо журнал, інакше зошит заросте
        pass

    def reply(self, body, ctype="text/html; charset=utf-8", code=200):
        raw = body.encode("utf-8")
        self.send_response(code)
        self.send_header("Content-Type", ctype)
        self.send_header("Content-Length", str(len(raw)))
        self.end_headers()
        self.wfile.write(raw)

    def do_GET(self):
        server_hits["count"] += 1
        path, _, query = self.path.partition("?")
        params = dict(pair.split("=", 1) for pair in query.split("&") if "=" in pair)

        if path == "/page":
            number = int(params.get("n", 1))
            if not 1 <= number <= PAGE_COUNT:
                return self.reply("<h1>Немає такої сторінки</h1>", code=404)
            return self.reply(page_html(number, params.get("layout", "normal")))
        if path == "/dynamic":
            return self.reply(DYNAMIC_PAGE)
        if path == "/static/menu.js":
            return self.reply(MENU_JS, "application/javascript; charset=utf-8")
        if path == "/api/menu.json":
            return self.reply(json.dumps(MENU, ensure_ascii=False),
                              "application/json; charset=utf-8")
        if path == "/robots.txt":
            return self.reply(ROBOTS_TXT, "text/plain; charset=utf-8")
        return self.reply("<h1>Немає такої сторінки</h1>", code=404)


server = http.server.ThreadingHTTPServer(("127.0.0.1", 0), MenuHandler)  # 0 = вільний порт
threading.Thread(target=server.serve_forever, daemon=True).start()
BASE = "http://127.0.0.1:{}".format(server.server_address[1])

print("наш сервер працює на", BASE)

## 2 · Перший запит: що взагалі приїхало

Скрапінг починається з того самого, що й [тема 27](../27-http-requests/lecture.html):
звичайний GET. Різниця в тому, що тіло відповіді — не JSON, а HTML для людей.

In [ ]:
response = requests.get(BASE + "/page?n=1", timeout=5)

print("код стану  :", response.status_code)
print("тип вмісту :", response.headers["Content-Type"])
print("довжина    :", len(response.text), "символів")
print()
print(response.text[:260], "…")

## 3 · Чому не регулярками

[Тема 25](../25-regex/lecture.html) дала потужний інструмент, і спокуса величезна:
назва позиції лежить між `<h3 class="name">` і `</h3>`, шаблон пишеться за десять секунд.

Напишемо його — і прогонимо по чотирьох варіантах того самого пункту меню. Усі чотири
для браузера означають **одне й те саме**. Для регулярки — ні.

In [ ]:
VARIANTS = [
    ("як ми звикли",        '<h3 class="name">Капучино</h3>'),
    ("пробіли в атрибуті",  '<h3 class = "name">Капучино</h3>'),
    ("тег на двох рядках",  '<h3 class="name">\n  Капучино\n</h3>'),
    ("вкладений тег",       '<h3 class="name">Капучино <span class="new">нове</span></h3>'),
]

name_pattern = re.compile(r'<h3 class="name">(.*?)</h3>')

for label, snippet in VARIANTS:
    by_regex = name_pattern.findall(snippet)
    by_parser = [tag.get_text(strip=True)
                 for tag in BeautifulSoup(snippet, "html.parser").select(".name")]
    print("{:<20} регулярка: {:<40} парсер: {}".format(
        label, repr(by_regex[0]) if by_regex else "нічого не знайшла", by_parser))

Прочитай вивід ще раз. Регулярка справилась рівно з одним варіантом із чотирьох.
І найгірший рядок — останній: вона нічого не зламала, вона **тихо повернула сміття**
разом із розміткою всередині. Такий скрапер не падає — він просто пише в CSV
`Капучино <span class="new">нове</span>`.

Парсер знайшов тег у всіх чотирьох випадках і в жодному не повернув розмітку.
Зафіксуємо це перевіркою.

In [ ]:
def names_by_regex(snippet):
    """Наївний спосіб: шукаємо текст між відкривним і закривним тегом."""
    return name_pattern.findall(snippet)


def names_by_parser(snippet):
    """Бібліотечний спосіб: віддаємо розбір парсеру."""
    return [tag.get_text(strip=True)
            for tag in BeautifulSoup(snippet, "html.parser").select(".name")]


# на перших трьох варіантах парсер дає буквально ту саму відповідь
parsed = [names_by_parser(snippet) for _, snippet in VARIANTS]
assert parsed[0] == parsed[1] == parsed[2] == ["Капучино"], "парсер мав дати те саме!"

# регулярка збігається з парсером лише на першому варіанті
matches = [names_by_regex(snippet) == names_by_parser(snippet) for _, snippet in VARIANTS]
print("регулярка збіглася з парсером:", matches)
assert matches == [True, False, False, False], "очікували три розбіжності з чотирьох"
print("✅ HTML не регулярна мова — шаблон розсипається на першій же дрібниці")

Останній варіант вартий окремої уваги. Парсер повернув `'Капучинонове'` — два шматки
тексту склеїлись без пробілу. Це не поломка: `strip=True` обрізає пробіли **в кожному**
шматку окремо, а потім склеює їх упритул. Якщо всередині тега трапляються вкладені
теги, потрібен роздільник.

In [ ]:
nested = VARIANTS[3][1]
tag = BeautifulSoup(nested, "html.parser").select_one(".name")

print("get_text(strip=True)      :", repr(tag.get_text(strip=True)))
print('get_text(" ", strip=True) :', repr(tag.get_text(" ", strip=True)))
print()
print("Різниця важлива там, де всередині назви трапляється <span>, <b> або <br>.")

## 4 · BeautifulSoup: `find` і `find_all`

`BeautifulSoup(text, "html.parser")` перетворює рядок на **дерево**. Другий аргумент —
який саме парсер узяти. `html.parser` приїхав разом із Python і нічого не потребує;
`lxml` швидший, але це окремий пакет. Для навчання і для більшості задач вистачає першого.

`find` повертає **перший** відповідний тег або `None`, `find_all` — список усіх.

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

print("тип обʼєкта :", type(soup).__name__)
print("заголовок   :", soup.find("h1").get_text(strip=True))
print("статей      :", len(soup.find_all("article")))
print()

first_article = soup.find("article")
print("перша стаття цілком:")
print(first_article.prettify())

`find_all` уміє шукати не лише за іменем тега, а й за атрибутами. Клас — особливий
випадок: слово `class` у Python зайняте, тому пишеться `class_`.

In [ ]:
print("за іменем тега h3      :", len(soup.find_all("h3")))
print("за класом name         :", len(soup.find_all(class_="name")))
print("h3 з класом name       :", len(soup.find_all("h3", class_="name")))
print("тегів <span> будь-яких :", len(soup.find_all("span")))
print()
print("назви:", [tag.get_text(strip=True) for tag in soup.find_all(class_="name")])

## 5 · CSS-селектори: `select` і `select_one`

Той самий пошук, але мовою селекторів — тією, якою верстальник описує сторінку.
`select` повертає список, `select_one` — перший збіг або `None`.

Порівняємо два записи одного й того самого запиту. Другий коротший саме тоді, коли
умова складна: «елемент із класом `name` **усередині** блоку з класом `menu`».

In [ ]:
print(".name                    :", len(soup.select(".name")))
print("h3.name                  :", len(soup.select("h3.name")))
print(".menu .name              :", len(soup.select(".menu .name")))   # будь-який нащадок
print(".menu > article > h3     :", len(soup.select(".menu > article > h3")))  # прямий нащадок
print("#menu-root               :", len(soup.select("#menu-root")))    # такого id тут немає
print()

# те саме двома способами — результат мусить збігтися
by_find = [tag.get_text(strip=True) for tag in soup.find_all("h3", class_="name")]
by_select = [tag.get_text(strip=True) for tag in soup.select("h3.name")]
assert by_find == by_select, "find_all і select мали дати те саме!"
print("find_all == select :", by_find)
print("✅ це два записи одного пошуку, а не два різні механізми")

## 6 · Текст і атрибути — і головна пастка

Тег — не рядок. Щоб дістати текст, є `.text` (усе підряд, як є) і
`.get_text(strip=True)` (з обрізаними пробілами по краях). Різницю добре видно саме
на «Флет-вайті» з другої сторінки.

Атрибути беруться як зі словника. І тут той самий вибір, що й у словнику
з [теми 09](../09-dictionaries/lecture.html): `tag["href"]` кидає `KeyError`, якщо
атрибута немає, а `tag.get("href")` тихо повертає `None`.

In [ ]:
page_two = BeautifulSoup(requests.get(BASE + "/page?n=2", timeout=5).text, "html.parser")

flat_white_price = page_two.select_one(".price")
print(".text              :", repr(flat_white_price.text))
print(".get_text(strip)   :", repr(flat_white_price.get_text(strip=True)))
print()

for article in page_two.select("article.item"):
    name = article.select_one(".name").get_text(strip=True)
    more = article.select_one("a.more")
    print("{:<15} href через .get() : {}".format(name, more.get("href")))

Останній рядок — посилання без адреси. `get` повернув `None` і програма поїхала далі.
А тепер той самий тег через квадратні дужки.

In [ ]:
broken_link = page_two.select("a.more")[-1]      # у «Чаю» атрибута href немає
print("сам тег:", broken_link)

try:
    print(broken_link["href"])
except KeyError as error:
    print("tag[\"href\"] кинув KeyError:", error)

print('tag.get("href")           :', broken_link.get("href"))
print('tag.get("href", "немає")  :', broken_link.get("href", "немає"))

Обидві поведінки правильні — питання в тому, чого ти хочеш. Якщо посилання
**обовʼязкове** й без нього запис безглуздий, квадратні дужки корисні: скрапер
зупиниться там, де щось пішло не так. Якщо посилання необовʼязкове — `get`
із запасним значенням.

Мовчазний `None`, який поїхав далі й потрапив у CSV, — найгірший із трьох варіантів.

## 7 · Одна сторінка → список словників

Мета скрапінгу — не «розібрати HTML», а отримати **таблицю**. Тому парсер віддає
не теги, а звичайні словники: далі з ними працює все, що ми вже вміємо.

Функція навмисно не намагається бути розумною. Вона робить одне: перекладає теги
в поля. Нормалізацією займемось окремо, бо це інша задача.

Один вибір тут принциповий. **Назва обовʼязкова**: позиція меню без назви — не
позиція, і якщо її немає, краще впасти, ніж записати порожній рядок у звіт.
А **ціни може не бути** — це нормальний стан справ, і його ми обробляємо.

In [ ]:
def parse_page(html_text):
    """Перетворює HTML сторінки меню на список словників."""
    page = BeautifulSoup(html_text, "html.parser")
    rows = []
    for article in page.select(".menu article.item"):
        price_tag = article.select_one(".price")
        rows.append({
            # назва обовʼязкова: якщо тега немає, хай краще буде гучна помилка
            "name": article.select_one(".name").get_text(strip=True),
            # а ціни може не бути — тоді select_one поверне None, і ми кладемо None
            "price_raw": price_tag.get_text(strip=True) if price_tag else None,
            "cat": article.select_one(".cat").get_text(strip=True),
            # посилання необовʼязкове, тому .get(), а не квадратні дужки
            "link": article.select_one("a.more").get("href"),
        })
    return rows


for row in parse_page(requests.get(BASE + "/page?n=2", timeout=5).text):
    print(row)

Ось вона, вся правда про сторінку: у «Рафа» `price_raw` дорівнює `None`, у «Чаю»
`link` теж `None`, а ціна записана як `30,00 UAH`. Якби ми написали
`article.select_one(".price").get_text()` без перевірки, зошит впав би з
`AttributeError: 'NoneType' object has no attribute 'get_text'` — саме на «Рафі»,
на пʼятій позиції з восьми.

## 8 · Пагінація: іти за посиланням «далі»

Сторінок три, але скрапер не повинен цього знати. Він робить те саме, що людина:
дочитав сторінку — пошукав посилання «далі» — пішов за ним. Немає посилання — кінець.

Одразу закладемо дві речі, про які йшлося в лекції: **паузу** між запитами й
**кеш** сторінок, щоб повторний запуск не смикав сервер за тим самим.

In [ ]:
page_cache = {}                 # адреса → текст сторінки


def fetch(url, pause=0.2):
    """Бере сторінку з кешу, а якщо її там немає — з сервера, з паузою."""
    if url in page_cache:
        return page_cache[url]
    time.sleep(pause)           # не смикаємо сервер частіше, ніж треба
    text = requests.get(url, timeout=5,
                        headers={"User-Agent": "kavyarnia-lesson/1.0"}).text
    page_cache[url] = text
    return text


def crawl(start_url, limit=20):
    """Іде за посиланням «далі», поки воно є. limit — щоб не зациклитись."""
    rows, url, visited = [], start_url, 0
    while url and visited < limit:
        html_text = fetch(url)
        rows += parse_page(html_text)
        visited += 1
        next_link = BeautifulSoup(html_text, "html.parser").select_one("a.next")
        # немає посилання «далі» — сторінка була остання
        url = BASE + next_link["href"] if next_link else None
    return rows, visited


hits_before = server_hits["count"]
menu_rows, pages_seen = crawl(BASE + "/page?n=1")

print("сторінок пройдено:", pages_seen)
print("позицій зібрано  :", len(menu_rows))
print("запитів до сервера:", server_hits["count"] - hits_before)
print()
for row in menu_rows:
    print("  {:<15} {!r:<12} {}".format(row["name"], row["price_raw"], row["cat"]))

`limit=20` тут не формальність. Якщо на сторінці колись зʼявиться посилання «далі»,
що веде саму на себе, цикл `while` крутитиметься вічно й довбатиме сервер. Стеля
кількості сторінок — найдешевший запобіжник, який можна написати.

## 9 · «Код відпрацював» ≠ «дані придатні»

Скрапер не впав і зібрав вісім рядків. Здається, перемога. Насправді ми ще нічого
не знаємо: у трьох рядках із восьми ціна не є числом.

Спочатку подивимось на дані чесно, а вже потім будемо їх лагодити.

In [ ]:
missing_price = [row["name"] for row in menu_rows if row["price_raw"] is None]
missing_link = [row["name"] for row in menu_rows if row["link"] is None]
# «звичайною» вважаємо ціну виду «45 грн» — число, пробіл, гривні
usual = re.compile(r"^\d+ грн$")
odd_price = [row["name"] for row in menu_rows
             if row["price_raw"] is not None and not usual.match(row["price_raw"])]

print("усього позицій        :", len(menu_rows))
print("без ціни              :", missing_price)
print("без посилання         :", missing_link)
print("ціна в іншому вигляді :", odd_price)
print()
print("частка придатних без обробки: {}/{}".format(
    len(menu_rows) - len(missing_price) - len(odd_price), len(menu_rows)))

Пʼять із восьми. Якби ми довірливо написали `int(row["price_raw"].split()[0])`,
скрапер упав би на «Рафі», а до «Чаю» навіть не дійшов би.

Тепер нормалізація. Правило одне: **числа дістаємо регуляркою, а не парсером**.
Це не суперечить розділу 3 — там регулярка програла на розмітці, а тут вона працює
з уже витягнутим текстом, тобто зі звичайним рядком. Це саме та задача, для якої
вона й створена.

In [ ]:
number_pattern = re.compile(r"\d+(?:[.,]\d+)?")


def price_to_number(raw):
    """З «  55  грн » і «30,00 UAH» робить число. З None — None."""
    if raw is None:
        return None
    found = number_pattern.search(raw)
    if not found:
        return None
    # кома як десятковий роздільник — звична для українських цінників
    return float(found.group(0).replace(",", "."))


for sample in ["25 грн", "  55  грн ", "30,00 UAH", None, "за домовленістю"]:
    print("{!r:<18} → {}".format(sample, price_to_number(sample)))

assert price_to_number("  55  грн ") == 55.0
assert price_to_number("30,00 UAH") == 30.0
assert price_to_number(None) is None
print("✅ нормалізація впоралась із усіма трьома нерівностями")

In [ ]:
clean_rows = []
for row in menu_rows:
    clean = dict(row)                       # не псуємо те, що зібрали
    clean["price"] = price_to_number(row["price_raw"])
    clean_rows.append(clean)

with_price = [row for row in clean_rows if row["price"] is not None]
average = sum(row["price"] for row in with_price) / len(with_price)

print("рядків усього      :", len(clean_rows))
print("рядків із ціною    :", len(with_price))
print("середня ціна       : {:.1f} грн".format(average))
print()
print("УВАГА: середня порахована по {} позиціях із {} — «Раф» у неї не входить.".format(
    len(with_price), len(clean_rows)))

## 10 · У CSV

Далі — те, що ми вже вміємо з [теми 22](../22-csv-json/lecture.html). Список
словників лягає в `DictWriter` без жодних зусиль. Позицію без ціни залишаємо
з порожньою коміркою: викидати рядок мовчки — гірше, ніж зберегти прогалину.

In [ ]:
work_dir = Path(tempfile.mkdtemp(prefix="scraping-"))
csv_path = work_dir / "menu.csv"

with open(csv_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["name", "price", "cat", "link"])
    writer.writeheader()
    for row in clean_rows:
        writer.writerow({key: row[key] for key in ("name", "price", "cat", "link")})

print("файл:", csv_path)
print()
print(csv_path.read_text(encoding="utf-8"))

In [ ]:
with open(csv_path, newline="", encoding="utf-8") as file:
    read_back = list(csv.DictReader(file))

assert len(read_back) == len(MENU), "у файл мали потрапити всі позиції"
assert read_back[4]["price"] == "", "у «Рафа» ціна має бути порожньою, а не вигаданою"
print("рядків у файлі:", len(read_back))
print("«Раф» у файлі :", read_back[4])
print("✅ таблиця з HTML-сторінок зібрана")

## 11 · Чому скрапер ламається

А тепер найважливіше. Уяви, що на сайті зробили редизайн. Наш сервер уміє віддавати
ту саму сторінку зі зміненою версткою — параметром `layout`.

Порівняємо два селектори:

* **крихкий** `.menu > article > h3` — тримається за те, що `<h3>` є **прямим**
  нащадком `<article>`;
* **стійкий** `.menu .name` — тримається за клас, який щось означає.

In [ ]:
LAYOUTS = [
    ("нічого не міняли",   "normal"),
    ("додали обгортку",    "wrap"),
    ("перейменували клас", "rename"),
]

FRAGILE = ".menu > article > h3"
STABLE = ".menu .name"

for label, layout in LAYOUTS:
    page = BeautifulSoup(fetch(BASE + "/page?n=1&layout=" + layout), "html.parser")
    print("{:<20} крихкий: {}   стійкий: {}".format(
        label, len(page.select(FRAGILE)), len(page.select(STABLE))))

Зайва обгортка вбила крихкий селектор і не зачепила стійкий. Перейменований клас —
навпаки. Висновок не «використовуй класи», а суворіший: **безсмертного селектора
немає**. Верстка зміниться, і колись зміниться саме так, як ти не передбачив.

Тому питання не «як не зламатись», а «як **помітити**, що зламався». Мовчазний
порожній список — найгірший результат: скрапер відпрацював, звіт зібрався, у ньому нуль
рядків, і ніхто цього не читає до кінця кварталу.

In [ ]:
def parse_page_checked(html_text, expected_at_least=1):
    """Те саме, що parse_page, але голосно скаржиться на порожній результат."""
    rows = parse_page(html_text)
    if len(rows) < expected_at_least:
        raise ValueError(
            "сторінка розібралась у {} позицій, а очікували щонайменше {} — "
            "схоже, верстка змінилась".format(len(rows), expected_at_least))
    return rows


# на нормальній верстці перевірка мовчить
print("normal:", len(parse_page_checked(fetch(BASE + "/page?n=1"), 3)), "позицій")

# зайва обгортка нашому parse_page байдужа: він шукає нащадків, а не прямих дітей
print("wrap  :", len(parse_page_checked(fetch(BASE + "/page?n=1&layout=wrap"), 3)), "позицій")

# а перейменований клас ламає вже сам parse_page — і робить це гучно
try:
    parse_page_checked(fetch(BASE + "/page?n=1&layout=rename"), 3)
except AttributeError as error:
    print("rename: AttributeError —", error)

# найгірший сценарій: зник контейнер .menu — тегів не знайшлось узагалі
broken_html = fetch(BASE + "/page?n=1").replace('class="menu"', 'class="menu-list"')
print()
print("без перевірки поверне мовчазний нуль:", len(parse_page(broken_html)))
try:
    parse_page_checked(broken_html, 3)
except ValueError as error:
    print("з перевіркою:", error)

Три різні поломки — три різні розвʼязки.

Зайва обгортка нашому `parse_page` не зашкодила: він шукає `.menu article.item`
і `.name` **серед нащадків**, а не серед прямих дітей. Крихкий селектор із таблиці
вище на цьому й впав.

Перейменований клас ламає `parse_page` на рівень глибше, ніж ми готувались: статті
`article.item` нікуди не поділись, тому до перевірки «чи знайшлось хоч щось» справа
не дійшла — `select_one(".name")` повернув `None`, і `None.get_text()` кинув
`AttributeError`. Це добре: помилка гучна. Погано було б написати
`if name_tag else ""` і мовчки отримати вісім позицій із порожніми назвами.

А от зниклий контейнер — саме той випадок, заради якого перевірка й існує. Без неї
скрапер відпрацював би до кінця, повернув порожній список і записав у CSV самий
заголовок. З перевіркою — зупинився на місці й сказав, що саме не так.

In [ ]:
def quality_report(rows):
    """Скільки полів реально заповнено — коротка звірка після кожного збирання."""
    return {
        "рядків": len(rows),
        "з назвою": sum(1 for row in rows if row["name"]),
        "з ціною": sum(1 for row in rows if row["price_raw"]),
        "з посиланням": sum(1 for row in rows if row["link"]),
    }


print("нормальна верстка:", quality_report(menu_rows))
print()
print("Такий звіт варто друкувати після кожного запуску: якщо «з назвою» раптом стало 0,")
print("це видно одразу, а не через квартал.")

## 12 · Динамічна сторінка: чому список порожній

Тепер сторінка `/dynamic`. У браузері вона виглядає так само — те саме меню, ті самі
вісім позицій. Спробуємо розібрати її нашим кодом.

In [ ]:
dynamic_html = requests.get(BASE + "/dynamic", timeout=5).text
print(dynamic_html)

dynamic_soup = BeautifulSoup(dynamic_html, "html.parser")
print("знайдено позицій:", len(dynamic_soup.select(".name")))
print("порожній контейнер:", dynamic_soup.select_one("#menu-root"))

Нуль. І це не помилка нашого коду: у тексті, який приїхав, позицій справді немає.
Є **порожній** `<div id="menu-root">` і тег `<script>`. Вміст домальовує JavaScript
уже в браузері, після завантаження, — а `requests` JavaScript не виконує. Він
завантажувач, а не браузер.

Що робити? Спочатку — найдешевше: подивитись, **звідки** сторінка бере дані.
У браузері це вкладка «Мережа»; у нас — просто прочитати той самий скрипт.

In [ ]:
script_tag = dynamic_soup.select_one("script")
script_url = BASE + script_tag["src"]
script_text = requests.get(script_url, timeout=5).text

print("скрипт лежить тут:", script_url)
print(script_text)

# адреси в лапках усередині скрипта — перше, куди варто подивитись
found_urls = re.findall(r'["\'](/[^"\']+\.json)["\']', script_text)
print("схоже на адресу даних:", found_urls)

In [ ]:
hidden_api = requests.get(BASE + found_urls[0], timeout=5)
items = hidden_api.json()

print("тип вмісту:", hidden_api.headers["Content-Type"])
print("позицій   :", len(items))
print("перша     :", items[0])
print()
print("Це вже JSON — розбирати HTML не потрібно взагалі.")
print("Динамічна сторінка виявилась не перешкодою, а підказкою, де лежить справжнє API.")

Ось чому [тема 28](../28-working-with-api/lecture.html) стоїть **перед** цією.
Дуже часто «сторінка з JavaScript» означає, що API вже є — просто його ніхто не
назвав публічним. Коли така адреса знаходиться, скрапінг закінчується, не почавшись:
далі звичайна робота з JSON.

Коли не знаходиться — лишається браузерна автоматизація: **Playwright** або
**Selenium** піднімають справжній браузер, чекають на JS і віддають готову сторінку.
Це працює, але коштує на порядок дорожче: сотні мегабайтів, секунди на сторінку
замість мілісекунд і ще одна річ, яка ламається. У цьому курсі ми їх не вивчаємо —
важливо знати, що вони існують і коли до них тягнутись.

## 13 · `robots.txt`

Файл `/robots.txt` лежить у корені сайту й адресований роботам. Це звичайний текст,
який читається одним запитом.

In [ ]:
robots = requests.get(BASE + "/robots.txt", timeout=5)
print("код стану:", robots.status_code)
print(robots.text)

disallowed = [line.split(":", 1)[1].strip()
              for line in robots.text.splitlines()
              if line.lower().startswith("disallow:")]
print("закриті розділи:", disallowed)
print("наш шлях /page під забороною:", any("/page".startswith(rule) for rule in disallowed))

Зверни увагу: сервер віддав цей файл звичайним двохсотим кодом і нічого не заборонив
технічно. `robots.txt` — **прохання**, а не замок. Ми могли б проігнорувати рядок
`Disallow: /item` і сходити туди — сервер не заперечив би.

Саме тому це питання не технічне. Технічно можна майже все; питання в тому, як
ти поводишся з чужим сервісом. І `Crawl-delay: 1` тут теж не декорація: власник
прямо просить не частіше ніж раз на секунду.

## 14 · Ввічливість: кеш і пауза

Наш `fetch` уже кешує сторінки. Перевіримо, що це не просто гарні слова: зберемо
меню вдруге й подивимось на лічильник запитів сервера.

In [ ]:
hits_before = server_hits["count"]
start = time.perf_counter()
second_run, _ = crawl(BASE + "/page?n=1")
elapsed = time.perf_counter() - start

print("позицій зібрано вдруге :", len(second_run))
print("нових запитів до сервера:", server_hits["count"] - hits_before)
print("часу витрачено          : {:.3f} с".format(elapsed))
assert second_run == menu_rows, "з кешу мали прийти ті самі дані"
print()
print("✅ той самий результат, нуль запитів, миттєво")

Це найдешевша ввічливість, яку взагалі можна проявити: **не питати двічі те саме**.
Поки ти налагоджуєш селектори, сторінка потрібна тобі десятки разів — і жоден із цих
разів не мусить доходити до чужого сервера.

Порахуймо ще й ціну паузи, щоб вона не була абстракцією.

In [ ]:
REQUEST_COST = 0.05        # скільки приблизно триває один запит, секунд
PAGES = 200                # уявний сайт на двісті сторінок

for pause in (0.0, 0.2, 0.5, 1.0):
    per_second = 1 / (REQUEST_COST + pause)
    total_minutes = PAGES * (REQUEST_COST + pause) / 60
    verdict = "так тебе заблокують" if per_second > 5 else "нормальне навантаження"
    print("пауза {:.1f} с → {:5.1f} запитів/с · усе збирання {:4.1f} хв · {}".format(
        pause, per_second, total_minutes, verdict))

Подивись на другий рядок: пауза 0.2 с розтягує збирання з 10 секунд до 50 — і при
цьому знімає навантаження впʼятеро. Півхвилини різниці для тебе, істотна різниця
для чужого сервера. Нульова пауза не означає «швидко»: вона означає «до першого
блокування», після якого збирання триватиме не 10 секунд, а ніколи.

## 15 · Прибираємо за собою

Зупиняємо сервер і видаляємо тимчасову теку. Після цієї клітинки в системі не
лишиться нічого від зошита.

In [ ]:
server.shutdown()
server.server_close()
shutil.rmtree(work_dir)

print("сервер зупинено, тимчасова тека видалена")
print("тека існує:", work_dir.exists())
print()
print("Усього сервер віддав відповідей:", server_hits["count"])

## Що далі — три рівні

**🟢 Рівень 1.** Додай у `parse_page` ще одне поле — `id` позиції, витягнутий із
посилання `href="/item?id=4"`. Підказка: `href.split("=")[-1]`, але не забудь, що
в однієї позиції `href` дорівнює `None`.

**🟡 Рівень 2.** Напиши функцію `report(rows)`, яка друкує середню ціну **окремо по
категоріях** — і чесно показує, скільки позицій у кожній категорії довелось
пропустити через відсутню ціну.

**🔴 Рівень 3.** Додай серверу четвертий варіант верстки, у якому ціна лежить не
в `<span class="price">`, а в атрибуті: `<article class="item" data-price="45">`.
Навчи `parse_page` брати ціну звідти, якщо тега `.price` немає, — і переконайся
перевіркою, що на старій верстці нічого не зламалось.

Докладніші завдання — у [homework.html](homework.html).